# Week 2 — The Context Lab

Last week you found that *how you ask* changes *what you get*. This week you build the
thing that does the asking, one part at a time, and watch each part earn its place.

**How to work through this.** Run the cells in order. Before most calls you will see the
exact context your code assembled and what it costs in tokens — read that string. It is
the whole subject of the course, printed.

**On the rate limit.** The free tier allows about **6 generation calls per minute**, in a
fixed window. This notebook makes roughly 18. It will pause; that is the quota, not a
bug. `counttokens()` and `show()` are free, so inspect as much as you like.

**Switching providers.** Everything below runs on Gemini by default. Set
`LLM_PROVIDER=lightning` in `.env` to run this exact notebook against Lightning AI
instead -- no cell below needs to change. See `llm_client.py` for what does and does
not carry over between the two (`counttokens()` in particular is a free exact count on
Gemini and a rough estimate on Lightning).

In [1]:
from context_lab import ask, show, counttokens, calls

---
## Part 1 — Tokens, on your own corpus

The lecture's table was measured on prepared sentences. This one is measured on *your*
project. Replace the text below with two or three real sentences from the domain you
proposed in Week 1 — in the language your corpus is actually written in.

In [6]:
MY_TEXT = "Модель може працювати лише з тим, що є в її контексті."

tokens = counttokens(MY_TEXT)
words = len(MY_TEXT.split())
print(f"chars={len(MY_TEXT)}  words={words}  tokens={tokens}  tok/word={tokens/words:.2f}")

chars=54  words=11  tokens=13  tok/word=1.18


In [3]:
# The same meaning in English, for comparison. Replace with your own translation.
EN = "The model can only work with what is in its context."
print(f"Ukrainian: {counttokens(MY_TEXT):>3} tokens")
print(f"English:   {counttokens(EN):>3} tokens")
print("\nNow try: a proper name from your domain, a long number, an identifier.")

Ukrainian:  13 tokens
English:    13 tokens

Now try: a proper name from your domain, a long number, an identifier.


Run `python tokens.py "your own text"` in the terminal for anything else you want to
measure. It is free, and it is the same tokeniser your `usage_metadata` reports.

**Write down your tokens-per-word figure.** Every cost estimate you make in Weeks 4, 10
and 11 starts from it.

### ✍️ Your notes

**What I changed:**

**What changed in the answer:**

**Why I think it changed:**

---
## Part 2 — The three counters

`thoughts_token_count` has been in your terminal output since Week 1. Here is what it
does.

Two calls, same trivial prompt. Watch the **thinking** number, not the answer.

In [11]:
r1 = ask("Reply with exactly: setup complete", show_prompt=False, label="run 1")


=== run 1 =======================================================

setup complete

  [tokens] prompt=14 thinking=0 answer=11   [calls this session: 5]


In [12]:
r2 = ask("Reply with exactly: setup complete", show_prompt=False, label="run 2")

# 0, not None -- llm_client coerces a missing count. See Part 6.
t1 = r1.usage.thinking_tokens
t2 = r2.usage.thinking_tokens
print(f"\nthinking: run 1 = {t1}, run 2 = {t2}, difference = {abs(t1 - t2)}")
print("Answer tokens both times: 2. These thinking tokens are billed at output rates.")


=== run 2 =======================================================

setup complete

  [tokens] prompt=14 thinking=128 answer=11   [calls this session: 6]

thinking: run 1 = 0, run 2 = 128, difference = 128
Answer tokens both times: 2. These thinking tokens are billed at output rates.


Two things to notice, and they are the reason this section exists:

1. The model spent tokens *reasoning* about a prompt with nothing to reason about.
2. The two counts are **not the same**. You cannot budget this by assuming a number.

### ✍️ Your notes

**What I changed:**

**What changed in the answer:**

**Why I think it changed:**

---
## Part 3 — Temperature

Experiment 5 last week: the same prompt, three times, three different answers. Here is
the control that explains it.

The first pair below repeats that experiment at the default temperature. The second pair
switches to a closed-form question at `temperature=0`: an open-ended question is not
guaranteed to produce identical text even at `temperature=0`, because Gemini's batched
serving is not perfectly reproducible, and a narrow, single-answer prompt gives
floating-point jitter nowhere to flip the outcome.

In [14]:
a = ask("Name one interesting fact about graph databases.", show_prompt=False, label="default a")
b = ask("Name one interesting fact about graph databases.", show_prompt=False, label="default b")
print(f"\nidentical? {a.text.strip() == b.text.strip()}")


=== default a ===================================================

One interesting fact: Graph databases model relationships as first-class citizens, allowing fast traversals and queries like shortest path or neighborhood analysis because they store direct connections (index-free adjacency) rather than relying on expensive joins.

  [tokens] prompt=16 thinking=64 answer=51   [calls this session: 9]

=== default b ===================================================

Fact: Graph databases treat relationships as first-class citizens and store them as edges between nodes, enabling fast, native traversal of complex networks (often with index-free adjacency, so traversing a linked path is O(1) per hop).

  [tokens] prompt=16 thinking=64 answer=56   [calls this session: 10]

identical? False


In [17]:
c = ask("List 15 words that rhyme with 'flight'", temperature=0,
        show_prompt=False, label="temperature=0 a")
d = ask("List 15 words that rhyme with 'flight'", temperature=0,
        show_prompt=False, label="temperature=0 b")
print(f"\nidentical? {c.text.strip() == d.text.strip()}")


=== temperature=0 a =============================================

- light
- night
- right
- might
- tight
- sight
- bright
- blight
- slight
- plight
- alight
- outright
- overnight
- moonlight
- sunlight

  [tokens] prompt=18 thinking=832 answer=56   [calls this session: 15]

=== temperature=0 b =============================================

- bright
- light
- night
- sight
- tight
- fight
- might
- right
- bite
- cite
- mite
- height
- fright
- plight
- write

  [tokens] prompt=18 thinking=448 answer=53   [calls this session: 16]

identical? False


The model produces a probability distribution over next tokens; temperature rescales it
before sampling. At 0 the highest-ranked candidate is taken every time. This is why the
comparison above switches to a closed-form question rather than reusing the open-ended
one from Experiment 5: an open-ended prompt has several near-tied candidates, and
floating-point non-determinism in Gemini's batched serving can still select different
tokens at `temperature=0`. A narrow, single-answer prompt removes that room to diverge.

**Answer this before moving on:** in the system you proposed in Week 1, which calls need
a value near zero? Classification, extraction, routing, and anything whose output your
code will parse.

### ✍️ Your notes

**What I changed:**

**What changed in the answer:**

**Why I think it changed:**

---
## Part 4 — The context ladder

**This is the session.**

One question. Six rounds. Each round adds exactly one part of a context, and nothing else
changes. The question is the one from Week 1's Experiment 4 — the one the model invented
an answer to.

| Round | Added | Changes between calls? |
|---|---|---|
| 0 | nothing — the bare question | — |
| 1 | `system=` — standing rules | never |
| 2 | `task=` — what to do | never |
| 3 | `sources=` — the passages | **every call** |
| 4 | `fmt=` — the shape of the answer | never |
| 5 | `examples=` — demonstrations | never |

Read each printed context before you read each answer.

In [3]:
QUESTION = "What is assessed in Lab 2 of the Modern AI Systems course, and what is it worth?"

SYSTEM = (
    "You answer questions about the Modern AI Systems course using only the sources "
    "you are given. If the sources do not answer the question, reply exactly: not stated. "
    "Never fill a gap from your own knowledge. Make answers as short and concise as possible."
)

TASK = "Answer the question using only the SOURCES above."

SOURCES = [
    "Lab 2 - Knowledge graph and Graph RAG - due end of Week 7 - 25% of the final "
    "grade. Deliverables: Neo4j database and graph model; re-seedable ingest.py; "
    "Cypher queries; graph or hybrid retrieval implementation; measured comparison "
    "against the Lab 1 system on the shared gold set; README with AI-use disclosure.",

    "Lab 2 marking: graph model quality 20%, Neo4j/Cypher implementation and "
    "re-seedable ingest 20%, retrieval strategy 20%, vector-versus-graph analysis on "
    "the gold set 15%, oral walkthrough 15%, documentation 10%.",
]

FORMAT = (
    "Three bullet points maximum. Put the source number in square brackets after each "
    "claim. No preamble, no closing sentence."
)

EXAMPLES = [
    ("What is assesed in Lab 1?", "Lab 1 tests understanding of tokens, temperature and context [1]."),
    ("What is Lab 3 worth?", "- Lab 3 is worth 25% of the final grade [1]."),
    ("When are the library opening hours?", "not stated"),
]

print("Parts defined. Nothing sent yet.")

Parts defined. Nothing sent yet.


In [ ]:
from context_lab import ask, show, counttokens, calls

In [19]:
# Round 0 — the bare question. This is Week 1, Experiment 4, round one.
ask(QUESTION, label="round 0 - nothing")


=== round 0 - nothing ===========================================
QUESTION
What is assessed in Lab 2 of the Modern AI Systems course, and what is it worth?
------------------------------------------------------------------------
  [22 tokens (rough estimate -- see counttokens() docstring), every one of them paid for on every call]

I don’t have access to your specific course materials. Could you tell me which institution and term this “Modern AI Systems” course is (and ideally share a link or syllabus)? If you can’t share a link, I can still help you quickly find it.

In the meantime, you can check:
- The course syllabus (often a PDF) for Lab 2 section: description and weight/points.
- The course LMS (Canvas/Blackboard/Brightspace) under Assessments or Modules for “Lab 2” and its grading rubric.
- The lab worksheet or assignment brief for what you’re supposed to do, which usually states the points or percentage.
- Grading rubric or TA/instructor announcements for any updates.

If you 

ChatResult(text='I don’t have access to your specific course materials. Could you tell me which institution and term this “Modern AI Systems” course is (and ideally share a link or syllabus)? If you can’t share a link, I can still help you quickly find it.\n\nIn the meantime, you can check:\n- The course syllabus (often a PDF) for Lab 2 section: description and weight/points.\n- The course LMS (Canvas/Blackboard/Brightspace) under Assessments or Modules for “Lab 2” and its grading rubric.\n- The lab worksheet or assignment brief for what you’re supposed to do, which usually states the points or percentage.\n- Grading rubric or TA/instructor announcements for any updates.\n\nIf you paste the Lab 2 description or rubric here, I’ll summarize exactly what’s assessed and how many points it’s worth.', usage=Usage(prompt_tokens=28, thinking_tokens=128, answer_tokens=183), parsed=None, raw={'id': 'chatcmpl-EO0Jh15JdQ3nZstgl284j6HZFPPp2', 'object': 'chat.completion', 'created': 1789389797, 'mod

In [20]:
# Round 1 — + the standing rules. Note they are NOT in the prompt string.
ask(QUESTION, system=SYSTEM, label="round 1 - + system")


=== round 1 - + system ==========================================
SYSTEM (sent as a system instruction, not as prompt text)
You answer questions about the Modern AI Systems course using only the sources you are given. If the sources do not answer the question, reply exactly: not stated. Never fill a gap from your own knowledge.

QUESTION
What is assessed in Lab 2 of the Modern AI Systems course, and what is it worth?
------------------------------------------------------------------------
  [73 tokens (rough estimate -- see counttokens() docstring), every one of them paid for on every call]

not stated

  [tokens] prompt=73 thinking=64 answer=11   [calls this session: 18]


ChatResult(text='not stated', usage=Usage(prompt_tokens=73, thinking_tokens=64, answer_tokens=11), parsed=None, raw={'id': 'chatcmpl-EO0NfSWPcskiSKX4Yq3nxbxBakilD', 'object': 'chat.completion', 'created': 1789390043, 'model': 'gpt-5-nano-2025-08-07', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'not stated'}, 'finish_reason': 'stop', 'content_filter_results': {'hate': {'filtered': False}, 'self_harm': {'filtered': False}, 'sexual': {'filtered': False}, 'violence': {'filtered': False}, 'jailbreak': {'filtered': False, 'detected': False}, 'profanity': {'filtered': False, 'detected': False}}}], 'usage': {'prompt_tokens': 73, 'completion_tokens': 75, 'total_tokens': 148, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'completion_tokens_details': {'audio_tokens': 0, 'reasoning_tokens': 64, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'system_fingerprint': '', 'service_tier': 'default'})

In [21]:
# Round 2 — + the task.
ask(QUESTION, system=SYSTEM, task=TASK, label="round 2 - + task")


=== round 2 - + task ============================================
SYSTEM (sent as a system instruction, not as prompt text)
You answer questions about the Modern AI Systems course using only the sources you are given. If the sources do not answer the question, reply exactly: not stated. Never fill a gap from your own knowledge.

TASK
Answer the question using only the SOURCES above.

QUESTION
What is assessed in Lab 2 of the Modern AI Systems course, and what is it worth?
------------------------------------------------------------------------
  [87 tokens (rough estimate -- see counttokens() docstring), every one of them paid for on every call]

not stated

  [tokens] prompt=85 thinking=64 answer=11   [calls this session: 19]


ChatResult(text='not stated', usage=Usage(prompt_tokens=85, thinking_tokens=64, answer_tokens=11), parsed=None, raw={'id': 'chatcmpl-EO0OXUCOh2QjEawNfJV7AEi32KhQG', 'object': 'chat.completion', 'created': 1789390097, 'model': 'gpt-5-nano-2025-08-07', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'not stated'}, 'finish_reason': 'stop', 'content_filter_results': {'hate': {'filtered': False}, 'self_harm': {'filtered': False}, 'sexual': {'filtered': False}, 'violence': {'filtered': False}, 'jailbreak': {'filtered': False, 'detected': False}, 'profanity': {'filtered': False, 'detected': False}}}], 'usage': {'prompt_tokens': 85, 'completion_tokens': 75, 'total_tokens': 160, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'completion_tokens_details': {'audio_tokens': 0, 'reasoning_tokens': 64, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'system_fingerprint': '', 'service_tier': 'default'})

In [22]:
# Round 3 — + the sources. The only part that changes from question to question.
ask(QUESTION, system=SYSTEM, task=TASK, sources=SOURCES, label="round 3 - + sources")


=== round 3 - + sources =========================================
SYSTEM (sent as a system instruction, not as prompt text)
You answer questions about the Modern AI Systems course using only the sources you are given. If the sources do not answer the question, reply exactly: not stated. Never fill a gap from your own knowledge.

SOURCES
[1] Lab 2 - Knowledge graph and Graph RAG - due end of Week 7 - 25% of the final grade. Deliverables: Neo4j database and graph model; re-seedable ingest.py; Cypher queries; graph or hybrid retrieval implementation; measured comparison against the Lab 1 system on the shared gold set; README with AI-use disclosure.
[2] Lab 2 marking: graph model quality 20%, Neo4j/Cypher implementation and re-seedable ingest 20%, retrieval strategy 20%, vector-versus-graph analysis on the gold set 15%, oral walkthrough 15%, documentation 10%.

TASK
Answer the question using only the SOURCES above.

QUESTION
What is assessed in Lab 2 of the Modern AI Systems course, and w

ChatResult(text='- Graph model quality — 20%\n- Neo4j/Cypher implementation and re-seedable ingest — 20%\n- Retrieval strategy — 20%\n- Vector-versus-graph analysis on the gold set — 15%\n- Oral walkthrough — 15%\n- Documentation — 10%\n\nTotal: 100% of Lab 2 marks.', usage=Usage(prompt_tokens=222, thinking_tokens=192, answer_tokens=81), parsed=None, raw={'id': 'chatcmpl-EO0PTIVt2ghEvEhV2bi7A5nTOGHD6', 'object': 'chat.completion', 'created': 1789390155, 'model': 'gpt-5-nano-2025-08-07', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '- Graph model quality — 20%\n- Neo4j/Cypher implementation and re-seedable ingest — 20%\n- Retrieval strategy — 20%\n- Vector-versus-graph analysis on the gold set — 15%\n- Oral walkthrough — 15%\n- Documentation — 10%\n\nTotal: 100% of Lab 2 marks.'}, 'finish_reason': 'stop', 'content_filter_results': {'hate': {'filtered': False}, 'self_harm': {'filtered': False}, 'sexual': {'filtered': False}, 'violence': {'filtered': False}, 'jailb

In [23]:
# Round 4 — + the output format.
ask(QUESTION, system=SYSTEM, task=TASK, sources=SOURCES, fmt=FORMAT,
    label="round 4 - + format")


=== round 4 - + format ==========================================
SYSTEM (sent as a system instruction, not as prompt text)
You answer questions about the Modern AI Systems course using only the sources you are given. If the sources do not answer the question, reply exactly: not stated. Never fill a gap from your own knowledge.

SOURCES
[1] Lab 2 - Knowledge graph and Graph RAG - due end of Week 7 - 25% of the final grade. Deliverables: Neo4j database and graph model; re-seedable ingest.py; Cypher queries; graph or hybrid retrieval implementation; measured comparison against the Lab 1 system on the shared gold set; README with AI-use disclosure.
[2] Lab 2 marking: graph model quality 20%, Neo4j/Cypher implementation and re-seedable ingest 20%, retrieval strategy 20%, vector-versus-graph analysis on the gold set 15%, oral walkthrough 15%, documentation 10%.

TASK
Answer the question using only the SOURCES above.

FORMAT
Three bullet points maximum. Put the source number in square brack

ChatResult(text='- Assessed components for Lab 2: graph model quality, Neo4j/Cypher implementation and re-seedable ingest, retrieval strategy, vector-versus-graph analysis on the gold set, oral walkthrough, and documentation. [2]\n\n- Worth: Lab 2 accounts for 25% of the final grade. [1]', usage=Usage(prompt_tokens=248, thinking_tokens=320, answer_tokens=78), parsed=None, raw={'id': 'chatcmpl-EO0QptyZMNrqcMgs68IOkP1wq9HKN', 'object': 'chat.completion', 'created': 1789390239, 'model': 'gpt-5-nano-2025-08-07', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '- Assessed components for Lab 2: graph model quality, Neo4j/Cypher implementation and re-seedable ingest, retrieval strategy, vector-versus-graph analysis on the gold set, oral walkthrough, and documentation. [2]\n\n- Worth: Lab 2 accounts for 25% of the final grade. [1]'}, 'finish_reason': 'stop', 'content_filter_results': {'hate': {'filtered': False}, 'self_harm': {'filtered': False}, 'sexual': {'filtered': Fal

In [28]:
# Round 5 — + worked examples. Note what the second example teaches.
ask(QUESTION, system=SYSTEM, task=TASK, sources=SOURCES, fmt=FORMAT,
    examples=EXAMPLES, label="round 5 - + examples")


=== round 5 - + examples ========================================
SYSTEM (sent as a system instruction, not as prompt text)
You answer questions about the Modern AI Systems course using only the sources you are given. If the sources do not answer the question, reply exactly: not stated. Never fill a gap from your own knowledge. Make answers as short and concise as possible.

SOURCES
[1] Lab 2 - Knowledge graph and Graph RAG - due end of Week 7 - 25% of the final grade. Deliverables: Neo4j database and graph model; re-seedable ingest.py; Cypher queries; graph or hybrid retrieval implementation; measured comparison against the Lab 1 system on the shared gold set; README with AI-use disclosure.
[2] Lab 2 marking: graph model quality 20%, Neo4j/Cypher implementation and re-seedable ingest 20%, retrieval strategy 20%, vector-versus-graph analysis on the gold set 15%, oral walkthrough 15%, documentation 10%.

EXAMPLES
  What is assesed in Lab 1?  ->  Lab 1 tests understanding of tokens, tem

ChatResult(text='- Lab 2 assesses knowledge graph and Graph RAG work, including Neo4j model, re-seedable ingest, Cypher queries, graph or hybrid retrieval, and evaluation against the Lab 1 gold set. [1]\n- It is worth 25% of the final grade. [1]', usage=Usage(prompt_tokens=328, thinking_tokens=320, answer_tokens=71), parsed=None, raw={'id': 'chatcmpl-EO0VaNm0sj399JduqKO0QRDX3AGQA', 'object': 'chat.completion', 'created': 1789390534, 'model': 'gpt-5-nano-2025-08-07', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '- Lab 2 assesses knowledge graph and Graph RAG work, including Neo4j model, re-seedable ingest, Cypher queries, graph or hybrid retrieval, and evaluation against the Lab 1 gold set. [1]\n- It is worth 25% of the final grade. [1]'}, 'finish_reason': 'stop', 'content_filter_results': {'hate': {'filtered': False}, 'self_harm': {'filtered': False}, 'sexual': {'filtered': False}, 'violence': {'filtered': False}, 'jailbreak': {'filtered': False, 'detected': Fal

### What each round cost

`show()` and `counttokens()` make no API call, so this cell is free. It prints what you have
been paying for on every single call above.

In [29]:
from context_lab import render

rounds = [
    ("0  bare question", None, {}),
    ("1  + system", SYSTEM, {}),
    ("2  + task", SYSTEM, {"task": TASK}),
    ("3  + sources", SYSTEM, {"task": TASK, "sources": SOURCES}),
    ("4  + format", SYSTEM, {"task": TASK, "sources": SOURCES, "fmt": FORMAT}),
    ("5  + examples", SYSTEM, {"task": TASK, "sources": SOURCES, "fmt": FORMAT,
                               "examples": EXAMPLES}),
]

print(f"  {'round':<18}{'tokens':>8}{'added':>8}")
print("  " + "-" * 34)
previous = 0
for name, system, parts in rounds:
    total = counttokens(render(QUESTION, **parts)) + (counttokens(system) if system else 0)
    print(f"  {name:<18}{total:>8}{total - previous:>8}")
    previous = total

  round               tokens   added
  ----------------------------------
  0  bare question        22      22
  1  + system             85      63
  2  + task               99      14
  3  + sources           234     135
  4  + format            266      32
  5  + examples          325      59


**Three questions to answer from your own output:**

1. At which round did the answer become *correct*? At which did it become *usable*?
2. Which round added the most tokens, and is it the one that helped most?
3. Rounds 1, 2, 4 and 5 never change. Round 3 changes with every question. Which of those
   is the code you will be writing for the rest of the semester?

### ✍️ Your notes

**What I changed:**

**What changed in the answer:**

**Why I think it changed:**

---
## Part 5 — Asking for a shape

Experiment 3 last week asked politely for JSON and got it *sometimes*. A prompt is a
request. Here is the difference between requesting and constraining.

In [30]:
import json

loose = ask(
    "List 3 risks of committing an API key to a public repository. "
    "Return ONLY a JSON array of objects with keys 'risk' and 'severity'. "
    "No prose, no code fence.",
    show_prompt=False, label="asking nicely",
)

try:
    parsed = json.loads(loose.text)
    print(f"\nparsed cleanly: {len(parsed)} items")
except json.JSONDecodeError as exc:
    print(f"\nDID NOT PARSE: {exc}")
    print(f"starts with: {loose.text[:60]!r}")


=== asking nicely ===============================================

[
  {"risk": "Unauthorized access to services and data due to exposed API key", "severity": "critical"},
  {"risk": "Quota exhaustion or incident-based charges from misuse of the API key", "severity": "high"},
  {"risk": "Credential compromise enabling misuse of related services and reputational damage", "severity": "high"}
]

  [tokens] prompt=45 thinking=256 answer=81   [calls this session: 25]

parsed cleanly: 3 items


### ✍️ Your notes

**What I changed:**

**What changed in the answer:**

**Why I think it changed:**

---
## Part 6 — The thinking dial

`thinking_level` takes MINIMAL, LOW, MEDIUM or HIGH. (`thinking_budget` is rejected from
Gemini 3.5 onwards.) Both too little and too much are expensive, in different ways.

**This section requires `LLM_PROVIDER=gemini`.** It is the one part of this notebook that
does not carry over. Lightning accepts `reasoning_effort` but was measured not to honour
it: setting it to `none` produced *more* reasoning than `high` did. The dial will not move
on that backend.

Below is the short version — one character-counting question at the two ends of the dial.
For the full table across all four levels and both task types, run
`python thinking_levels.py` in the terminal. That is 8 more calls, so run it once and
compare with a neighbour rather than everyone running it at once.

In [2]:
# The model does not see characters (Part 1). Here is what that costs when the
# task IS characters. The correct answer is one line of Python -- see `correct`.
PUZZLE_TEXT = (
    "Великі мовні моделі не бачать окремих літер: вони працюють із токенами, "
    "і саме тому подібні завдання виявляються несподівано складними для них."
)
LETTER = "о"
correct = PUZZLE_TEXT.lower().count(LETTER)

PUZZLE = (
    f"How many times does the letter '{LETTER}' appear in the text below? "
    f"Count every occurrence. Answer with just the number, nothing else."
    f"\n\n{PUZZLE_TEXT}"
)

for level in ["MINIMAL", "HIGH"]:
    r = ask(PUZZLE, level=level, show_prompt=False, label=f"thinking_level={level}")
    print(f"  correct answer is {correct} (counted in Python) -- thinking tokens: "
          f"{r.usage.thinking_tokens}")


=== thinking_level=MINIMAL ======================================

9

  [tokens] prompt=81 thinking=640 answer=10   [calls this session: 1]
  correct answer is 9 (counted in Python) -- thinking tokens: 640

=== thinking_level=HIGH =========================================

9

  [tokens] prompt=81 thinking=832 answer=11   [calls this session: 2]
  correct answer is 9 (counted in Python) -- thinking tokens: 832


At MINIMAL the model does not reason at all: `thinking_tokens` comes back as 0, the answer
arrives in about a second, and it is **wrong** — typically off by one, stated with no
indication that anything went wrong. At HIGH the same question costs over a thousand
thinking tokens and several seconds, and the count is right.

This is the Part 1 claim made expensive. The model does not see characters, only tokens,
so counting a letter is not a lookup for it; it is a task that has to be worked through.
Notice which answer was the cheap, fast, confident one. That is the failure direction
people do not plan for.

Judging how much reasoning a call deserves is Week 11 material. Today it is enough to
know the setting exists, that the default is not always right, and that both directions
cost.

### ✍️ Your notes

**What I changed:**

**What changed in the answer:**

**Why I think it changed:**

---
## Part 7 — The exit clause

Your ladder is grounded. Now ask it something the sources genuinely do not contain.

In [4]:
UNANSWERED = "What is the late submission penalty for Lab 2?"

ask(UNANSWERED, system=SYSTEM, task=TASK, sources=SOURCES, fmt=FORMAT,
    examples=EXAMPLES, label="with the exit clause")


=== with the exit clause ========================================
SYSTEM (sent as a system instruction, not as prompt text)
You answer questions about the Modern AI Systems course using only the sources you are given. If the sources do not answer the question, reply exactly: not stated. Never fill a gap from your own knowledge. Make answers as short and concise as possible.

SOURCES
[1] Lab 2 - Knowledge graph and Graph RAG - due end of Week 7 - 25% of the final grade. Deliverables: Neo4j database and graph model; re-seedable ingest.py; Cypher queries; graph or hybrid retrieval implementation; measured comparison against the Lab 1 system on the shared gold set; README with AI-use disclosure.
[2] Lab 2 marking: graph model quality 20%, Neo4j/Cypher implementation and re-seedable ingest 20%, retrieval strategy 20%, vector-versus-graph analysis on the gold set 15%, oral walkthrough 15%, documentation 10%.

EXAMPLES
  What is assesed in Lab 1?  ->  Lab 1 tests understanding of tokens, tem

ChatResult(text='not stated', usage=Usage(prompt_tokens=319, thinking_tokens=64, answer_tokens=11), parsed=None, raw={'id': 'chatcmpl-EO0luVfQA0dbHab5QHuNpZyi8N2wK', 'object': 'chat.completion', 'created': 1789391546, 'model': 'gpt-5-nano-2025-08-07', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'not stated'}, 'finish_reason': 'stop', 'content_filter_results': {'hate': {'filtered': False}, 'self_harm': {'filtered': False}, 'sexual': {'filtered': False}, 'violence': {'filtered': False}, 'jailbreak': {'filtered': False, 'detected': False}, 'profanity': {'filtered': False, 'detected': False}}}], 'usage': {'prompt_tokens': 319, 'completion_tokens': 75, 'total_tokens': 394, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'completion_tokens_details': {'audio_tokens': 0, 'reasoning_tokens': 64, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'system_fingerprint': '', 'service_tier': 'default'})

In [5]:
# The same question, the same sources. One sentence removed from SYSTEM.
SYSTEM_NO_EXIT = (
    "You answer questions about the Modern AI Systems course using the sources "
    "you are given."
)

ask(UNANSWERED, system=SYSTEM_NO_EXIT, task=TASK, sources=SOURCES, fmt=FORMAT,
    label="without the exit clause")


=== without the exit clause =====================================
SYSTEM (sent as a system instruction, not as prompt text)
You answer questions about the Modern AI Systems course using the sources you are given.

SOURCES
[1] Lab 2 - Knowledge graph and Graph RAG - due end of Week 7 - 25% of the final grade. Deliverables: Neo4j database and graph model; re-seedable ingest.py; Cypher queries; graph or hybrid retrieval implementation; measured comparison against the Lab 1 system on the shared gold set; README with AI-use disclosure.
[2] Lab 2 marking: graph model quality 20%, Neo4j/Cypher implementation and re-seedable ingest 20%, retrieval strategy 20%, vector-versus-graph analysis on the gold set 15%, oral walkthrough 15%, documentation 10%.

TASK
Answer the question using only the SOURCES above.

FORMAT
Three bullet points maximum. Put the source number in square brackets after each claim. No preamble, no closing sentence.

QUESTION
What is the late submission penalty for Lab 2?
----

ChatResult(text='- The provided sources do not specify any late submission penalty for Lab 2. [1][2]', usage=Usage(prompt_tokens=214, thinking_tokens=128, answer_tokens=30), parsed=None, raw={'id': 'chatcmpl-EO0mAFLzqxOeIbaITqLRcxONBYSvn', 'object': 'chat.completion', 'created': 1789391562, 'model': 'gpt-5-nano-2025-08-07', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '- The provided sources do not specify any late submission penalty for Lab 2. [1][2]'}, 'finish_reason': 'stop', 'content_filter_results': {'hate': {'filtered': False}, 'self_harm': {'filtered': False}, 'sexual': {'filtered': False}, 'violence': {'filtered': False}, 'jailbreak': {'filtered': False, 'detected': False}, 'profanity': {'filtered': False, 'detected': False}}}], 'usage': {'prompt_tokens': 214, 'completion_tokens': 158, 'total_tokens': 372, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'completion_tokens_details': {'audio_tokens': 0, 'reasoning_tokens': 128, 'accepted_

Compare the two answers carefully. The second is fluent, confident, formatted correctly,
cited — and, if it said anything at all about a penalty, invented.

**No setting disables this.** What you have are: ground it, constrain the output, give it
an explicit exit, and measure it. The exit clause is the cheapest of the four and the one
most often left out.

### ✍️ Your notes

**What I changed:**

**What changed in the answer:**

**Why I think it changed:**

---
## Before you close this

**Keep your ladder.** Rounds 1, 2, 4 and 5 — your `SYSTEM`, `TASK`, `FORMAT` and
`EXAMPLES` — are the prompt skeleton you submit with Lab 1. Copy them into your project
now, with your own domain in place of this one.

**Notice what is not in it yet:** no tools, no memory, no conversation history, and
`SOURCES` still pasted in by hand. That last one is Week 3: finding the right passage
among thousands, without knowing in advance which one you need.

**Due at the end of this week:** Milestone 0, the Project Proposal — one page, with
evidence against all five gate criteria. Lab 1 is not accepted without an approved
proposal, and Week 3 opens ingestion and the gold set, after which a change of domain
costs you real work.

In [7]:
print(f"generation calls this session: {calls()}")

generation calls this session: 4
